In [104]:
!pip install python-docx

In [105]:
!pip install -q langchain_openai==0.0.2 faiss-cpu==1.7.4 openai==1.6.1 tiktoken==0.5.2 langchain_community==0.0.11 langchain==0.1.0

In [106]:
from google.colab import userdata
import requests
import numpy as np
import os
import io
import tempfile
import tiktoken
from docx import Document
from scipy.spatial.distance import cdist

In [107]:
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, CharacterTextSplitter
import re
import requests
import openai
from openai import OpenAI
import os
from langchain.docstore.document import Document
import tiktoken
import matplotlib.pyplot as plt
from google.colab import userdata

In [108]:
ID_FOLDER = userdata.get('ID_FOLDER')
OAuth_token = userdata.get('OAuth_token')

In [109]:
# URL для получения токена
URL = "https://iam.api.cloud.yandex.net/iam/v1/tokens"

# Получение IAM-токена (с помощью request)
import requests

headers = {"Content-Type": "application/json"}

data = {
    "yandexPassportOauthToken": OAuth_token
}

response = requests.post(URL, headers=headers, json=data)

IAM_TOKEN = response.json()["iamToken"]
expiresAt = response.json()["expiresAt"]

print(f'Ваш токен действителен до: {expiresAt}')

Ваш токен действителен до: 2024-05-07T22:43:07.716267891Z


In [110]:
# База знаний для Markdown сохранена в файле doc_markdown_v1.2.txt, его ID 17-_-kA8SVuAU5OM_2gkzGqw_lZ0Zc4yo
id_doc_markdown = '17-_-kA8SVuAU5OM_2gkzGqw_lZ0Zc4yo'

In [111]:
# функция для загрузки документа по doc_id из гугл драйв в текстовом формате
def load_doc_id_text(file_id):
    # Download the document as plain text
    response = requests.get(f'https://drive.google.com/uc?export=download&id={file_id}')
    response.raise_for_status()
    text = response.text

    return text

In [112]:
data_from_markdown = load_doc_id_text(id_doc_markdown)

In [113]:
data_from_markdown[-100:]

'я информации на оборотной стороне свидетельства о поверке при оформлении его на бумажном носителе).\n'

In [114]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
      """Возвращает количество токенов в строке"""
      encoding = tiktoken.get_encoding(encoding_name)
      num_tokens = len(encoding.encode(string))
      return num_tokens

def split_text(text, max_count):
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    fragments = markdown_splitter.split_text(text)

    # Подсчет токенов для каждого фрагмента
    fragment_token_counts = [num_tokens_from_string(fragment.page_content, "cl100k_base") for fragment in fragments]


    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_count,
        chunk_overlap=0,
        length_function=lambda x: num_tokens_from_string(x, "cl100k_base")
    )

    source_chunks = [
        Document(page_content=chunk, metadata=fragment.metadata)
        for fragment in fragments
        for chunk in splitter.split_text(fragment.page_content)
    ]

    # Подсчет токенов для каждого source_chunk
    source_chunk_token_counts = [num_tokens_from_string(chunk.page_content, "cl100k_base") for chunk in source_chunks]


    return source_chunks, fragments

In [115]:
source_chunks, fragments = split_text(data_from_markdown, 1000)
print("Общее количество чанков: ",len(source_chunks))
print("Первый чанк ", source_chunks[0])
print("Крайний чанк ", source_chunks[len(source_chunks)-1])

Общее количество чанков:  623
Первый чанк  page_content='Документ предоставлен КонсультантПлюс'
Крайний чанк  page_content='5. На оборотной стороне свидетельств о поверке при оформлении их на бумажном носителе или в свидетельствах о поверке, оформляемых в виде электронного документа, по заявлению владельцев средств измерений или лиц, представивших средства измерений на поверку, или по согласованию с ними может указываться дополнительная информация, относящаяся к средствам измерений, месту их установки, особенностям поверки, включая сведения о пломбах, предотвращающих доступ к местам настройки (регулировки) средств измерений, принадлежности средств измерений (сведения о владельцах средств измерений), а также информация о прилагаемых к свидетельству о поверке документах (при невозможности размещения информации на оборотной стороне свидетельства о поверке при оформлении его на бумажном носителе).' metadata={'Header 1': 'ТРЕБОВАНИЯ К СОДЕРЖАНИЮ СВИДЕТЕЛЬСТВА О ПОВЕРКЕ'}


In [116]:
source_chunks[0].page_content

'Документ предоставлен КонсультантПлюс'

In [117]:
doc_texts = []
for i in range(len(source_chunks)):
    text = f'"{source_chunks[i].page_content}",'
    doc_texts.append(text)

In [ ]:
doc_texts

In [ ]:
doc_uri = f"emb://{ID_FOLDER}/text-search-doc/latest"
query_uri = f"emb://{ID_FOLDER}/text-search-query/latest"

embed_url = "https://llm.api.cloud.yandex.net:443/foundationModels/v1/textEmbedding"
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {IAM_TOKEN}", "x-folder-id": f"{ID_FOLDER}"}

query_text = "Тест"

def get_embedding(text: str, text_type: str = "doc") -> np.array:
    query_data = {
        "modelUri": doc_uri if text_type == "doc" else query_uri,
        "text": text,
    }

    return np.array(
        requests.post(embed_url, json=query_data, headers=headers).json()["embedding"]
    )


query_embedding = get_embedding(query_text, text_type="query")
docs_embedding = [get_embedding(doc_text) for doc_text in doc_texts]

In [ ]:
#Embedding базы знаний
docs_embedding

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
#Сохраним массив в файл
np.save('docs_embedding.npy', docs_embedding)

In [ ]:
#загрузка массива embeddings
loadedArray= np.load('docs_embedding.npy')

In [ ]:
# Вычисляем косинусное расстояние
dist = cdist(query_embedding[None, :], docs_embedding, metric="cosine")

# Вычисляем косинусное сходство
sim = 1 - dist

# most similar doc text
print(doc_texts[np.argmax(sim)])

"Заключение по проверке результатов испытаний должно содержать выводы по оценке результатов испытаний (положительные или отрицательные). Результаты испытаний стандартных образцов считаются положительными, если полученные по результатам испытаний метрологические и технические характеристики стандартных образцов соответствуют заявленным, полученные результаты испытаний стандартных образцов удовлетворяют требованиям программы испытаний, программа испытаний, акт испытаний с протоколами испытаний, проект описания типа стандартных образцов оформлены в соответствии с требованиями настоящего Порядка. Результаты испытаний стандартных образцов считаются отрицательными, если полученные по результатам испытаний метрологические и технические характеристики стандартных образцов не соответствуют заявленным, полученные результаты испытаний стандартных образцов удовлетворяют не всем требованиям программы испытаний, программа испытаний, акт испытаний с протоколами испытаний, проект описания типа стандар

In [ ]:
sim

array([0.29875918, 0.21947941, 0.28375869, 0.20375093, 0.23605913,
       0.25533822, 0.26628382, 0.27440932, 0.21990501, 0.23051182,
       0.18090814, 0.23512862, 0.18656137, 0.28030818, 0.22543434,
       0.24356425, 0.24531406, 0.22538959, 0.20776341, 0.21072829,
       0.19429951, 0.23343791, 0.25134007, 0.21191308, 0.22498019,
       0.18739858, 0.21970642, 0.21829638, 0.22246122, 0.21819515,
       0.20188341, 0.24109581, 0.20724461, 0.2041052 , 0.23019251,
       0.2066759 , 0.24077457, 0.15644218, 0.20794367, 0.19706483,
       0.22002023, 0.22561203, 0.27172507, 0.27044143, 0.26703013,
       0.25271927, 0.22146005, 0.25073312, 0.2637981 , 0.26096043,
       0.24878023, 0.21254566, 0.24018718, 0.18300759, 0.23393308,
       0.18732073, 0.21568174, 0.19971916, 0.20064302, 0.16572112,
       0.23353438, 0.19534374, 0.2243532 , 0.1955611 , 0.21539353,
       0.21645243, 0.17600106, 0.21451257, 0.22025581, 0.21144822,
       0.23669687, 0.24052238, 0.27510825, 0.20601393, 0.26068

In [ ]:
#вывод максимальных k значений
import heapq
k=3
heapq.nlargest(k, enumerate(sim), key=lambda x: x[1])

[(379, 0.3417073917101976),
 (539, 0.3397698013917898),
 (283, 0.3397284353850174)]